# 07 — Figure Generation

Generates all publication figures.

In [ ]:
import sys
sys.path.insert(0, '..')
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.models.stella import build_stella, STELLAConfig
from src.visualization.plots import *
from pathlib import Path

Path('../results/figures').mkdir(parents=True, exist_ok=True)

# Build model for weight visualization
model = build_stella()

# Figure 1: Learned spectral weights
plot_spectral_weights(model, '../results/figures/spectral_weights.pdf')
print('Spectral weights figure saved')

In [ ]:
# Figure 2: Architecture diagram (text-based ASCII)
arch_diagram = '''
STELLA Architecture:

Input EEG (B, C, T)
       |
       +------------------+
       |                  |
  TemporalPatch       SpectralBand
  Embed (dw-conv)     Encoder (FFT+bands)
       |                  |
 (B, P, D)           (B, 5, D)
       |                  |
  ChannelSpatial         |
  Transformer (2L)       |
       |                  |
       +--GatedFusion-----+
              |
         (B, P, D)
              |
       [CLS] | fused
              |
        TemporalMamba
        (S3M × 3 layers)
              |
       (B, P+1, D)
              |
         CLS token
              |
         z (B, D)
              |
      Downstream Head
'''
print(arch_diagram)

In [ ]:
# Figure 3: S3M state evolution visualization
cfg = STELLAConfig(n_channels=22, segment_len=320, patch_size=20, patch_stride=10,
                    d_model=64, mamba_layers=1, mamba_d_state=8)
small_model = build_stella(cfg)

# Hook to capture S3M states
x = torch.randn(1, 22, 320)
with torch.no_grad():
    z = small_model(x)
    
print(f'Representation shape: {z.shape}')

# Parameter breakdown
counts = small_model.count_parameters()
for k, v in counts.items():
    print(f'  {k:30s}: {v:>8,}')